# Deaths of Despair — Layer 2: Flagging Engine

Flags significant findings: outlier states, correlations, regional divergence.

In [1]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import sys
sys.path.insert(0, "/Users/matt/data-adventures")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pipeline.config import load_config, get_data_processed_dir

PROJECT = "deaths-of-despair"
config = load_config(PROJECT)
processed_dir = get_data_processed_dir(config)

state_panel = pd.read_parquet(processed_dir / "state_panel.parquet")
national_panel = pd.read_parquet(processed_dir / "national_panel.parquet")

STATE_INDICATORS = [
    "year", "state_fips", "state_name", "state_abbr",
    "overdose_death_rate", "suicide_rate", "deaths_despair_rate",
    "manufacturing_employees_thousands", "unemployment_rate",
    "median_household_income", "poverty_pct",
    "region", "mfg_yoy_pct_change", "mfg_index_2000",
]
available = [c for c in STATE_INDICATORS if c in state_panel.columns]
state_panel = state_panel[available].copy()

print(f"Panel: {state_panel.shape[0]:,} rows | {int(state_panel.year.min())}–{int(state_panel.year.max())}")


Panel: 918 rows | 1999–2016


## Flag: States with highest deaths of despair rates (latest year)

In [2]:
latest_year = int(state_panel["year"].max())
latest = state_panel[state_panel["year"] == latest_year].copy()

# Z-score flagging
mean_rate = latest["deaths_despair_rate"].mean()
std_rate = latest["deaths_despair_rate"].std()
latest["ddr_zscore"] = (latest["deaths_despair_rate"] - mean_rate) / std_rate
flagged = latest[latest["ddr_zscore"] > 1.5].sort_values("deaths_despair_rate", ascending=False)
print(f"States with z-score > 1.5 (>{mean_rate + 1.5*std_rate:.1f} deaths per 100k):")
print(flagged[["state_abbr","deaths_despair_rate","overdose_death_rate","suicide_rate","region","ddr_zscore"]].to_string(index=False))


States with z-score > 1.5 (>51.7 deaths per 100k):
state_abbr  deaths_despair_rate  overdose_death_rate  suicide_rate                 region  ddr_zscore
        WV              71.3211              52.0211          19.3             Appalachia    3.489345
        NH              56.1902              38.9902          17.2                  Other    1.955384
        OH              53.3225              39.1225          14.2 Rust Belt / Appalachia    1.664658
        PA              52.5509              37.8509          14.7 Rust Belt / Appalachia    1.586434


## Flag: Fastest-rising states over the study period

In [3]:
first_year = int(state_panel["year"].min())
start = state_panel[state_panel["year"] == first_year][["state_abbr","deaths_despair_rate"]].rename(columns={"deaths_despair_rate": "ddr_start"})
end = state_panel[state_panel["year"] == latest_year][["state_abbr","deaths_despair_rate"]].rename(columns={"deaths_despair_rate": "ddr_end"})
change = start.merge(end, on="state_abbr").dropna()
change["absolute_rise"] = change["ddr_end"] - change["ddr_start"]
change["pct_rise"] = (change["ddr_end"] - change["ddr_start"]) / change["ddr_start"] * 100

print(f"States with largest absolute rise in deaths of despair ({first_year}→{latest_year}):")
print(change.nlargest(10, "absolute_rise")[["state_abbr","ddr_start","ddr_end","absolute_rise","pct_rise"]].round(1).to_string(index=False))

fig = px.bar(
    change.nlargest(15, "absolute_rise"),
    x="state_abbr", y="absolute_rise",
    color="absolute_rise", color_continuous_scale="Reds",
    title=f"States with Largest Rise in Deaths of Despair ({first_year}→{latest_year})",
    labels={"absolute_rise": "Rise (deaths per 100k)", "state_abbr": "State"},
)
fig.show()


States with largest absolute rise in deaths of despair (1999→2016):
state_abbr  ddr_start  ddr_end  absolute_rise  pct_rise
        WV       16.2     71.3           55.1     339.0
        NH       15.5     56.2           40.7     263.1
        OH       13.9     53.3           39.5     285.0
        PA       18.4     52.6           34.1     184.9
        KY       16.5     50.3           33.9     205.9
        DC       13.4     44.0           30.6     227.3
        MA       14.2     41.8           27.6     194.7
        RI       14.6     42.0           27.4     188.0
        ME       18.7     44.6           25.9     139.0
        IN       13.6     39.4           25.8     189.6


## Flag: Manufacturing job loss vs. deaths of despair correlation

In [4]:
# Compute mfg loss from 2000 baseline
base_2000 = state_panel[state_panel["year"] == 2000][["state_abbr","manufacturing_employees_thousands"]].rename(columns={"manufacturing_employees_thousands": "mfg_2000"})
panel_with_base = latest.merge(base_2000, on="state_abbr", how="left")
panel_with_base["mfg_loss_pct"] = (panel_with_base["manufacturing_employees_thousands"] - panel_with_base["mfg_2000"]) / panel_with_base["mfg_2000"] * 100

valid = panel_with_base.dropna(subset=["mfg_loss_pct","deaths_despair_rate"])
corr = valid[["mfg_loss_pct","deaths_despair_rate"]].corr().iloc[0,1]
print(f"Correlation: mfg job loss % vs deaths of despair rate: r={corr:.3f}")

fig = px.scatter(
    valid,
    x="mfg_loss_pct", y="deaths_despair_rate",
    text="state_abbr", color="region",
    trendline="ols",
    title=f"Manufacturing Job Loss vs Deaths of Despair ({latest_year})<br><sub>r = {corr:.3f}</sub>",
    labels={"mfg_loss_pct": "% Mfg Job Loss Since 2000", "deaths_despair_rate": "Deaths of Despair per 100k"},
)
fig.update_traces(textposition="top center", selector=dict(mode="markers+text"))
fig.show()


Correlation: mfg job loss % vs deaths of despair rate: r=-0.182


## Flag: Poverty & income divergence in high-despair states

In [5]:
if "median_household_income" in state_panel.columns:
    recent = state_panel[state_panel["year"] >= 2010].copy()
    corr_income = recent[["median_household_income","deaths_despair_rate"]].corr().iloc[0,1]
    corr_poverty = recent[["poverty_pct","deaths_despair_rate"]].corr().iloc[0,1]
    print(f"Correlation — median income vs deaths of despair: r={corr_income:.3f}")
    print(f"Correlation — poverty rate vs deaths of despair:  r={corr_poverty:.3f}")

    latest_w_income = latest.dropna(subset=["median_household_income","deaths_despair_rate"])
    fig = px.scatter(
        latest_w_income,
        x="median_household_income", y="deaths_despair_rate",
        text="state_abbr", color="region",
        trendline="ols",
        title=f"Median Household Income vs Deaths of Despair ({latest_year})",
        labels={"median_household_income": "Median Household Income ($)", "deaths_despair_rate": "Deaths of Despair per 100k"},
    )
    fig.update_traces(textposition="top center", selector=dict(mode="markers+text"))
    fig.show()


Correlation — median income vs deaths of despair: r=-0.142
Correlation — poverty rate vs deaths of despair:  r=0.099


## Flagging summary

In [6]:
print("=== FLAGGING SUMMARY ===")
print()
print(f"1. States with z-score > 1.5 on deaths of despair ({latest_year}):")
for _, row in flagged.iterrows():
    print(f"   {row['state_abbr']:3s}: {row['deaths_despair_rate']:.1f} per 100k (overdose: {row['overdose_death_rate']:.1f}, suicide: {row['suicide_rate']:.1f})")
print()
print(f"2. Fastest-rising states ({first_year}→{latest_year}):")
for _, row in change.nlargest(5,"absolute_rise").iterrows():
    print(f"   {row['state_abbr']:3s}: +{row['absolute_rise']:.1f} per 100k (+{row['pct_rise']:.0f}%)")
print()
print(f"3. Manufacturing correlation: r={corr:.3f} (negative = more job loss → more despair)")


=== FLAGGING SUMMARY ===

1. States with z-score > 1.5 on deaths of despair (2016):
   WV : 71.3 per 100k (overdose: 52.0, suicide: 19.3)
   NH : 56.2 per 100k (overdose: 39.0, suicide: 17.2)
   OH : 53.3 per 100k (overdose: 39.1, suicide: 14.2)
   PA : 52.6 per 100k (overdose: 37.9, suicide: 14.7)

2. Fastest-rising states (1999→2016):
   WV : +55.1 per 100k (+339%)
   NH : +40.7 per 100k (+263%)
   OH : +39.5 per 100k (+285%)
   PA : +34.1 per 100k (+185%)
   KY : +33.9 per 100k (+206%)

3. Manufacturing correlation: r=-0.182 (negative = more job loss → more despair)
